<a href="https://colab.research.google.com/github/AribaRazi/MachineLearning/blob/main/Rnn_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense

In [ ]:


sentences = [
 "I love this product",
 "This movie made me smile",
 "Service was friendly and quick",
 "Today felt bright and happy",
 "This is the best day",
 "Absolutely fantastic experience",
 "I enjoyed every single moment",
 "Great job, well done",
 "The food tasted delicious",
 "Totally recommend to everyone",
 "Very satisfied with results",
 "This worked better than expected",
 "Amazing quality and value",
 "Such a pleasant surprise",
 "I feel positive about this",
 "I hate this product",
 "This movie bored me",
 "Service was rude and slow",
 "Today was cold and lonely",
 "This is the worst day",
 "Terrible experience overall",
 "I regret buying this",
 "Very disappointed with results",
 "The food tasted awful",
 "Do not recommend this",
 "It broke after one use",
 "Not worth the money",
 "Utterly frustrating and annoying",
 "I feel negative about this",
 "Such a waste of time",
]
labels = [1]*15 + [0]*15
labels = np.array(labels)

In [ ]:
# Tokenization

In [ ]:
vocab_size = 2000
tok= Tokenizer(num_words=vocab_size, oov_token = "OOV")
tok.fit_on_texts(sentences)

seqs= tok.texts_to_sequences(sentences)
maxlen=max(len(s) for s in seqs)

In [ ]:
# padding
x=pad_sequences(seqs, maxlen=maxlen, padding="post", truncating="post")
y=labels

In [ ]:
print(tok.word_counts)
print(tok.word_index)

OrderedDict({'i': 6, 'love': 1, 'this': 11, 'product': 2, 'movie': 2, 'made': 1, 'me': 2, 'smile': 1, 'service': 2, 'was': 3, 'friendly': 1, 'and': 6, 'quick': 1, 'today': 2, 'felt': 1, 'bright': 1, 'happy': 1, 'is': 2, 'the': 5, 'best': 1, 'day': 2, 'absolutely': 1, 'fantastic': 1, 'experience': 2, 'enjoyed': 1, 'every': 1, 'single': 1, 'moment': 1, 'great': 1, 'job': 1, 'well': 1, 'done': 1, 'food': 2, 'tasted': 2, 'delicious': 1, 'totally': 1, 'recommend': 2, 'to': 1, 'everyone': 1, 'very': 2, 'satisfied': 1, 'with': 2, 'results': 2, 'worked': 1, 'better': 1, 'than': 1, 'expected': 1, 'amazing': 1, 'quality': 1, 'value': 1, 'such': 2, 'a': 2, 'pleasant': 1, 'surprise': 1, 'feel': 2, 'positive': 1, 'about': 2, 'hate': 1, 'bored': 1, 'rude': 1, 'slow': 1, 'cold': 1, 'lonely': 1, 'worst': 1, 'terrible': 1, 'overall': 1, 'regret': 1, 'buying': 1, 'disappointed': 1, 'awful': 1, 'do': 1, 'not': 2, 'it': 1, 'broke': 1, 'after': 1, 'one': 1, 'use': 1, 'worth': 1, 'money': 1, 'utterly': 1, '

In [ ]:
maxlen

5

In [ ]:
x[0]

array([ 3, 26,  2,  7,  0], dtype=int32)

# model creation

In [ ]:
embed_dim = 16
rnn_units = 8

In [ ]:
inp = Input(shape = (maxlen,),dtype= "int32",name='input')

X= Embedding(input_dim=vocab_size, output_dim = embed_dim,mask_zero=True, name = 'embed')(inp)


In [ ]:
rnn = SimpleRNN(units =rnn_units,return_sequences = True, name ='simple_rnn')
x_last =  rnn(X)
out =  Dense (1,activation = 'sigmoid',name='output')(x_last)

# model creation
model = Model(inputs = inp,outputs=out)
model.compile(optimizer = 'adam',loss='binary_crossentropy',metrics=['accuracy'])
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 5)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed (Embedding)   │ (None, 5, 16)     │     32,000 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 5)         │          0 │ input[0][0]       │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ (None, 5, 8)      │        200 │ embed[0][0],      │
│ (SimpleRNN)         │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 5, 1)      │          9 │ simple_rnn[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,209 (125.82 KB)

 Trainable params: 32,209 (125.82 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(X,y,epochs=25, batch_size = 8,verbose = 1)

TypeError: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'

In [ ]:
|